# Inspeção de chunks — RAG com MDs estruturais

Use este notebook pra:
1. Listar editais indexados
2. Ver todos os chunks de um edital **em ordem do texto** (cap_num, chunk_index)
3. Inspecionar metadados de cada chunk
4. Rodar queries e ver top-k com scores
5. Estatísticas de tamanho

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else '.')

from dotenv import load_dotenv
load_dotenv()

from ingestion.rag import get_chroma_collection, buscar_chunks
from database.db import get_connection
import pandas as pd

# pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.width', 180)

/home/julio/Documentos/tcc_GENAI/demo0/edital-assistant/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Editais indexados

In [2]:
conn = get_connection()
rows = conn.execute('SELECT id, orgao, numero_edital, cargo, arquivo_origem FROM editais').fetchall()
conn.close()
editais_df = pd.DataFrame([dict(r) for r in rows])
editais_df

,id,orgao,numero_edital,cargo,arquivo_origem
0,1,BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E ...,01/2024,Analista,20240812EditalRetificadoFINALBNDES12agostoBNDE...
1,2,COMISSÃO DE VALORES MOBILIÁRIOS,1/2024 CVM,Analista CVM,edital-1-2024-abertura-concurso-cvm.pdf
2,3,PETRÓLEO BRASILEIRO S.A. – PETROBRAS,1 – PETROBRAS/PSP RH 2021,Profissional Petrobras de Nível Superior Júnior,editalpetroleobrasileiro.pdf


## 2. Todos os chunks de um edital (ordem de texto)

In [3]:
EDITAL_ID = 1   # ajuste conforme seu DB

col = get_chroma_collection()
res = col.get(where={'edital_id': EDITAL_ID})

rows = []
for i in range(len(res['ids'])):
    m = res['metadatas'][i]
    rows.append({
        'chunk_id': res['ids'][i],
        'cap_num': m.get('cap_num'),
        'chunk_index': m.get('chunk_index'),
        'tipo': m.get('tipo'),
        'h1': (m.get('h1') or '')[:60],
        'h2': (m.get('h2') or '')[:60],
        'h3': (m.get('h3') or '')[:60],
        'numero_secao': m.get('numero_secao'),
        'pag': f"{m.get('pagina_inicio')}-{m.get('pagina_fim')}",
        'chars': len(res['documents'][i]),
        'texto': res['documents'][i][:120],
    })

chunks_df = pd.DataFrame(rows).sort_values(['cap_num', 'chunk_index']).reset_index(drop=True)
print(f'Total de chunks do edital {EDITAL_ID}: {len(chunks_df)}')
chunks_df

Total de chunks do edital 1: 83


,chunk_id,cap_num,chunk_index,tipo,h1,h2,h3,numero_secao,pag,chars,texto
0,edital_1_cap_0_chunk_0,0,0,preambulo,,,,,1-1,1065,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
1,edital_1_cap_1_chunk_0,1,0,corpo,1 - DAS DISPOSIÇÕES PRELIMINARES,1.1. - A Seleção Pública será regida por este ...,,1.1,1-2,1097,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
2,edital_1_cap_2_chunk_0,2,0,corpo,"2 - DO CARGO, ATRIBUIÇÕES DO CARGO, REMUNERAÇÃ...",2.1 - CARGO: ANALISTA,,2.1,2-4,1749,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
3,edital_1_cap_3_chunk_0,3,0,corpo,3 - DAS VAGAS RESERVADAS,3.1 - DAS VAGAS RESERVADAS ÀS PESSOAS COM DEFI...,3.1.2. - Do total de vagas ofertadas inicialme...,3.1.2.2,4-8,1184,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
4,edital_1_cap_3_chunk_1,3,1,corpo,3 - DAS VAGAS RESERVADAS,3.1 - DAS VAGAS RESERVADAS ÀS PESSOAS COM DEFI...,3.1.3. - Para se inscrever nesta Seleção Públi...,3.1.3,4-8,1948,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
...,...,...,...,...,...,...,...,...,...,...,...
78,edital_1_cap_15_chunk_0,15,0,anexo,ANEXO III - CRONOGRAMA,EVENTOS BÁSICOS - BNDES,,,46-47,3398,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
79,edital_1_cap_16_chunk_0,16,0,anexo,ANEXO IV - MODELO DE RELATÓRIO/LAUDO CARACTERI...,,,,47-47,1311,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
80,edital_1_cap_16_chunk_1,16,1,anexo,ANEXO IV - MODELO DE RELATÓRIO/LAUDO CARACTERI...,ATENÇÃO aos documentos e(ou) informações que d...,1 - Deficiência Auditiva,,47-47,631,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...
81,edital_1_cap_16_chunk_2,16,2,anexo,ANEXO IV - MODELO DE RELATÓRIO/LAUDO CARACTERI...,ATENÇÃO aos documentos e(ou) informações que d...,3 - Deficiência Intelectual e Deficiência Ment...,,47-47,657,Contexto: BANCO NACIONAL DE DESENVOLVIMENTO EC...


## 3. Ver o texto completo de um chunk específico

In [5]:
# Escolha uma linha do chunks_df acima
id=1
CHUNK_ID = chunks_df.iloc[id]['chunk_id']

idx = res['ids'].index(CHUNK_ID)
print('Metadata:')
for k, v in res['metadatas'][idx].items():
    print(f'  {k}: {v}')
print('\n' + '=' * 80 + '\n')
print(res['documents'][idx])

Metadata:
  edital_id: 1
  numero_edital: 01/2024
  cap_titulo: 1 - Das Disposições Preliminares
  pagina_inicio: 1
  h2: 1.1. - A Seleção Pública será regida por este Edital sob a responsabilidade da Fundação Cesgranrio e do BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES, e será constituída das seguintes etapas:
  numero_secao: 1.1
  orgao: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES
  h1: 1 - DAS DISPOSIÇÕES PRELIMINARES
  pagina_fim: 2
  h3: 
  arquivo_origem: 20240812EditalRetificadoFINALBNDES12agostoBNDES.pdf
  tipo: corpo
  chunk_index: 0
  cap_num: 1
  h4: 


Contexto: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES > 1 - DAS DISPOSIÇÕES PRELIMINARES > 1.1. - A Seleção Pública será regida por este Edital sob a responsabilidade da Fundação Cesgranrio e do BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES, e será constituída das seguintes etapas:

a) 1ª Etapa - Avaliação de Conhecimentos, mediante a aplicação de provas objetiv

## 4. Testar retrieval com uma query

In [6]:
QUERY = 'conteúdo programático de ciência de dados'
TOP_K = 10
FILTRO_EDITAL = EDITAL_ID    # ou None pra buscar em todos

chunks = buscar_chunks(QUERY, n_results=TOP_K, edital_id=FILTRO_EDITAL)
print(f'{len(chunks)} chunks retornados.\n')

for i, c in enumerate(chunks):
    m = c['metadata']
    print(f"#{i+1} dist={c['distancia']:.4f} cap_{m['cap_num']} chunk_{m['chunk_index']} "
          f"secao={m['numero_secao']!r}")
    print(f"    h1={m['h1']!r}")
    print(f"    h2={m['h2']!r}")
    print(f"    h3={m['h3']!r}")
    print(f"    texto (primeiros 200 chars): {c['texto'][:200]!r}")
    print()

Carregando BGE-M3 (primeira vez pode demorar)...
10 chunks retornados.

#1 dist=0.4116 cap_14 chunk_12 secao=''
    h1='ANEXO II - CONTEÚDOS PROGRAMÁTICOS'
    h2='CONHECIMENTOS ESPECÍFICOS'
    h3='ANALISTA – CIÊNCIA DE DADOS'
    texto (primeiros 200 chars): 'Contexto: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES > ANEXO II - CONTEÚDOS PROGRAMÁTICOS > CONHECIMENTOS ESPECÍFICOS > ANALISTA – CIÊNCIA DE DADOS > V - GESTÃO DE PROJETOS DE CIÊNCI'

#2 dist=0.4511 cap_14 chunk_11 secao=''
    h1='ANEXO II - CONTEÚDOS PROGRAMÁTICOS'
    h2='CONHECIMENTOS ESPECÍFICOS'
    h3='ANALISTA – CIÊNCIA DE DADOS'
    texto (primeiros 200 chars): 'Contexto: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES > ANEXO II - CONTEÚDOS PROGRAMÁTICOS > CONHECIMENTOS ESPECÍFICOS > ANALISTA – CIÊNCIA DE DADOS > IV- DADOS E BASES DE DADOS\n\n1. C'

#3 dist=0.4534 cap_14 chunk_19 secao=''
    h1='ANEXO II - CONTEÚDOS PROGRAMÁTICOS'
    h2='CONHECIMENTOS ESPECÍFICOS'
    h3='ANALISTA – CI

In [17]:
QUERY = 'Qual o cronograma?'
TOP_K = 10
FILTRO_EDITAL = 1   # ou None pra buscar em todos

chunks = buscar_chunks(QUERY, n_results=TOP_K, edital_id=FILTRO_EDITAL)
print(f'{len(chunks)} chunks retornados.\n')

for i, c in enumerate(chunks):
    m = c['metadata']
    print(f"#{i+1} dist={c['distancia']:.4f} cap_{m['cap_num']} chunk_{m['chunk_index']} "
          f"secao={m['numero_secao']!r}")
    print(f"    h1={m['h1']!r}")
    print(f"    h2={m['h2']!r}")
    print(f"    h3={m['h3']!r}")
    print(f"    texto: {c['texto']}")
    print()

10 chunks retornados.

#1 dist=0.5866 cap_14 chunk_3 secao=''
    h1='ANEXO II - CONTEÚDOS PROGRAMÁTICOS'
    h2='CONHECIMENTOS BÁSICOS'
    h3='CONHECIMENTOS TRANSVERSAIS'
    texto: Contexto: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES > ANEXO II - CONTEÚDOS PROGRAMÁTICOS > CONHECIMENTOS BÁSICOS > CONHECIMENTOS TRANSVERSAIS > 3. Clima, Sustentabilidade e Responsabilidade Socioambiental e Climática

3.1 Meio Ambiente e Sustentabilidade

3.2 Política Nacional de Meio Ambiente (PNMA - Lei nº 6938/1981 e suas alterações)

3.2.1 Licenciamento ambiental – Portal Nacional de Licenciamento Ambiental (PNLA)

3.2.2 Sistema Nacional de Unidades de Conservação da Natureza (SNUC – Lei nº 9.985/2000 e suas alterações)

3.2.3 Lei sobre a Proteção da Vegetação Nativa (conhecida como Novo Código Florestal - Lei nº 12.651/2012 e suas alterações)

3.3 Clima e Sustentabilidade

3.3.1 Mudanças climáticas

3.3.2 Riscos físicos e de transição

3.3.3 Mitigação e adaptação

3.3.4 Transição e

## 5. Estatísticas de tamanho

In [7]:
chunks_df['chars'].describe()

count      83.000000
mean      999.265060
std       460.405126
min       456.000000
25%       635.000000
50%       957.000000
75%      1185.000000
max      3398.000000
Name: chars, dtype: float64

In [8]:
# Distribuição por cap_num
chunks_df.groupby('cap_num').agg(n_chunks=('chunk_id', 'count'), total_chars=('chars', 'sum'))

,n_chunks,total_chars
cap_num,,
0,1,1065
1,1,1097
2,1,1749
3,6,6816
6,9,9453
8,11,12977
9,21,14299
10,5,3171
13,1,479


## 6. Ver um chunk inteiro pelo (cap_num, chunk_index)

In [9]:
CAP = 16
IDX = 0

mask = (chunks_df['cap_num'] == CAP) & (chunks_df['chunk_index'] == IDX)
if not mask.any():
    print(f'Chunk cap_{CAP} idx_{IDX} não encontrado')
else:
    row = chunks_df[mask].iloc[0]
    doc_idx = res['ids'].index(row['chunk_id'])
    print(res['documents'][doc_idx])

Contexto: BANCO NACIONAL DE DESENVOLVIMENTO ECONÔMICO E SOCIAL - BNDES > ANEXO IV - MODELO DE RELATÓRIO/LAUDO CARACTERIZADOR DE DEFICIÊNCIA

MODELO DE RELATÓRIO/ LAUDO CARACTERIZADOR DE DEFICIÊNCIA PARA A INSCRIÇÃO (candidatos(as) que se declararam com deficiência).

Atesto, para fins de participação em Seleção Pública, que ________________________________________________ 
_________________________________, portador(a) do documento de identidade nº ______________________, "é considerada pessoa com deficiência à luz da legislação brasileira que apresentar o(s) seguinte(s) impedimento(s) físicos, auditivos, visuais, intelectuais ou psicossociais/mentais"______________________________________________________________________________________________, que resulta(m) no comprometimento das seguintes funções/funcionalidades __________________________________________________ 
__________________________________. Informo, ainda, a provável causa do comprometimento ________________________________

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [3]:
from database.db import get_connection
import json
conn = get_connection()
row = conn.execute("SELECT dados_json FROM editais WHERE id = 3").fetchone()
conn.close()
dados = json.loads(row["dados_json"])
print("Cronograma extraído:")
for c in dados.get("cronograma", []):
    print(f"  {c}")
print()
print(f"Total de eventos: {len(dados.get('cronograma', []))}")

Cronograma extraído:
  {'evento': 'Aplicação das provas objetivas', 'data': '20/02/2022'}

Total de eventos: 1
